# Sales Onboarding Transformer - Colab Smoke Test

This notebook builds the first Transformer model for next-field prediction using the prepared #18 transformer_inputs arrays.

In [ ]:
import json
import numpy as np
import tensorflow as tf
from pathlib import Path

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

base_dir = Path('/content/SALES-ONBOARDING-GUIDANCE-SYSTEM-AI-')
config_path = base_dir / 'data' / 'transformer_inputs' / 'preprocessing_config.json'

with open(config_path, 'r', encoding='utf-8') as f:
    config = json.load(f)

print(config)

In [ ]:
base_dir = Path('/content/SALES-ONBOARDING-GUIDANCE-SYSTEM-AI-')
data_dir = base_dir / 'data' / 'transformer_inputs'

train_x = np.load(data_dir / 'train_inputs.npy').astype(np.int32)
train_mask = np.load(data_dir / 'train_masks.npy').astype(bool)
train_y = np.load(data_dir / 'train_targets.npy').astype(np.int32) - 1

val_x = np.load(data_dir / 'validation_inputs.npy').astype(np.int32)
val_mask = np.load(data_dir / 'validation_masks.npy').astype(bool)
val_y = np.load(data_dir / 'validation_targets.npy').astype(np.int32) - 1

print('train_x:', train_x.shape)
print('train_mask:', train_mask.shape)
print('train_y:', train_y.shape)
print('valid labels min/max:', train_y.min(), train_y.max())

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class PositionalEmbedding(layers.Layer):
    def __init__(self, max_sequence_length, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_sequence_length = max_sequence_length
        self.position_embedding = self.add_weight(
            name='position_embedding',
            shape=(max_sequence_length, embedding_dim),
            initializer='uniform',
            trainable=True,
        )

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.position_embedding[:seq_len]


class TransformerBlock(layers.Layer):
    def __init__(self, embedding_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads,
            dropout=dropout_rate,
        )
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embedding_dim),
            layers.Dropout(dropout_rate),
        ])
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs, attention_mask=None):
        attention_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
        )
        attention_output = self.dropout1(attention_output)
        output = self.norm1(inputs + attention_output)
        ffn_output = self.ffn(output)
        ffn_output = self.dropout2(ffn_output)
        return self.norm2(output + ffn_output)

MAX_LEN = config['max_sequence_length']
VOCAB_SIZE = config['vocabulary_size'] + 1
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2

inputs = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input_ids')
mask = layers.Input(shape=(MAX_LEN,), dtype=tf.bool_, name='attention_mask')

x = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=D_MODEL,
    mask_zero=True,
    name='token_embedding'
)(inputs)

x = PositionalEmbedding(MAX_LEN, D_MODEL)(x)
attention_mask_4d = tf.cast(mask[:, tf.newaxis, tf.newaxis, :], tf.bool)

for _ in range(NUM_BLOCKS):
    x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, dropout_rate=0.1)(x, attention_mask=attention_mask_4d)

last_index = tf.reduce_sum(tf.cast(mask, tf.int32), axis=1) - 1
pooled = tf.gather(x, last_index, batch_dims=1, axis=1)
outputs = layers.Dense(33, activation='softmax', name='next_field_probabilities')(pooled)

model = tf.keras.Model(inputs=[inputs, mask], outputs=outputs, name='onboarding_transformer')
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name='acc'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')
    ],
)

history = model.fit(
    (train_x, train_mask),
    train_y,
    validation_data=((val_x, val_mask), val_y),
    epochs=2,
    batch_size=64,
    verbose=1,
)

print('Training finished. This is the first smoke test for the Transformer.')